<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/Basic_python_colab_and_fasta_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python and AI in Google Colab — Building and Testing a FASTA Tool
**MICR 4203/5203 • Fall 2027 • 20 points • 60–75 minutes**

No programming experience required. Learn to run and modify Python; recognize a variable, loop, and function; direct an AI assistant; test a known result; and explain your reasoning.

Save your own copy in Drive and rename it `Lastname_Firstname_FASTA.ipynb`. Run cells in order. Double-click Markdown answer areas to edit. Setup and tests are provided; your AI-generated function belongs only in Part C. An unfinished notebook runs without crashing, but reports **NOT READY** until you supply working code. That status is not a passing result.

Your first AI prompt is allowed to fail. Finding an error, checking it, and improving the prompt is excellent evidence of learning. No screenshots or separate report required.

## A. Your first Python experiments — 4 points (10–15 min)
Run A1, then change `sequence` to `"AAAA"` and rerun. A variable stores a value; `len` and `count` are functions that return results.

In [ ]:
sequence = "ATGCGGGTA"
print("Sequence:", sequence)
print("Length:", len(sequence))
print("Number of G bases:", sequence.count("G"))

### A1 answers
- What changed? **[Your answer]**
- What does `len(sequence)` calculate? **[Your answer]**
- What does `sequence.count("G")` calculate? **[Your answer]**

### A2. A loop
Run this cell, add a fourth DNA sequence to the list, and rerun.

In [ ]:
sequences = ["ATGC", "GGCCAA", "ATATATAT"]
for dna in sequences:
    print(dna, len(dna))

### A2 answer
What does the loop do, and how many times does it run after your change? **[Your answer]**

## B. The biological problem — 3 points (10 min)
FASTA records have a header starting with `>` and one or more sequence lines. Some tools use only the first word of the header as the identifier. Both headers below could become `sample`.

```text
>sample A | gene=psbA
ATGCGT
AATT
>sample A | gene=psbD
GGCC
ATAT
```
Predict the cleaned output before running your tool:
```text
>seq001
ATGCGT
AATT
>seq002
GGCC
ATAT
```
### B answers
1. Why do we need unique identifiers? **[Your answer]**
2. Why must sequences stay unchanged? **[Your answer]**
3. Why keep a table connecting new IDs to original headers? **[Your answer]**

### Provided setup — run once before C
All inputs are synthetic. No upload, package installation, or Drive mount is needed. Paths identify files inside the runtime; runtime files are temporary. Setup rewrites only the supplied synthetic inputs, never your code or answers.

In [ ]:
from pathlib import Path
import csv, json, zipfile, tempfile, hashlib

ROOT = Path.cwd() / "fasta_lab_v3"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
DATA.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
INPUT = DATA / "example.fasta"
CLEANED = RESULTS / "cleaned_sequences.fasta"
MAPPING = RESULTS / "header_mapping.csv"
EXAMPLE = b">sample A | gene=psbA\nATGCGT\nAATT\n>sample A | gene=psbD\nGGCC\nATAT\n"
INPUT.write_bytes(EXAMPLE)
(DATA / "challenge.fasta").write_bytes(b'>same header, "quoted"\nacgTNn\nAA\n>same header, "quoted"\nGGcc\nTT')
(DATA / "gc_test.fasta").write_text(">test\nGCNN\n", encoding="utf-8")
validation = {"status": "NOT READY", "checks": []}
print(INPUT.read_text())
print("Input:", INPUT)
print("Output:", CLEANED)
print("Mapping:", MAPPING)

## C. Use AI to build a FASTA tool — 6 points (20–25 min)
Ask ChatGPT, Gemini, Claude, or another assistant for **one** function:

```python
clean_fasta(input_path, output_path, mapping_path)
```
Requirements:
- Use only Python's standard library and beginner-readable code.
- Replace headers in order with `>seq001`, `>seq002`, … (minimum three digits).
- Preserve every non-header sequence line exactly: case, line wrapping, and final newline presence. Do not strip or join sequence lines.
- Write a UTF-8 CSV with columns `new_id,original_header`. Store the full original header **without** `>` or its line ending; retain punctuation and spaces. Use `csv.writer` for quoting.
- Do not alter the input. Reject paths that would overwrite the input or use the same path for both outputs, before writing anything.

Starter prompt: “I am a beginning Python student studying bioinformatics in Google Colab. Help me write a readable function called clean_fasta(input_path, output_path, mapping_path) that replaces FASTA headers with seq001, seq002, etc., preserves sequence lines exactly, and writes a CSV linking each new ID to its original header. Do not overwrite my input. Use the standard library and explain each major section.” Add the precise requirements above.

Paste your function below, replacing the starter. A function is a reusable set of instructions; defining it does not run it.

In [ ]:
# Replace this starter with your AI-generated function.
def clean_fasta(input_path, output_path, mapping_path):
    raise NotImplementedError("Paste your AI-generated function here.")

Ask AI: “Explain how this function recognizes FASTA headers and assigns new names, as though I have never programmed before.”

### C explanation
In your own words, explain one part of your code in 2–3 sentences: **[Your answer]**

## D. Test and improve — 5 points (15 min)
### D1. Provided validation
Run this cell after defining your function. It runs the known two-record example, checks exact output and CSV rows, and checks the input is unchanged. Failure messages are evidence to investigate, not a grade. Rerun after revisions. Tests are strong checks for these inputs, not proof of correctness for every possible FASTA file.

In [ ]:
def verify_case(function, source, expected_headers):
    # Independent expected result; no FASTA parser shared with student code.
    with tempfile.TemporaryDirectory() as folder:
        base = Path(folder)
        src, dst, table = [base / name for name in ("in.fasta", "out.fasta", "map.csv")]
        src.write_bytes(source)
        expected_lines, index = [], 0
        for line in source.splitlines(keepends=True):
            if line.startswith(b">"):
                index += 1
                expected_lines.append(f">seq{index:03d}\n".encode())
            else:
                expected_lines.append(line)
        try:
            function(src, dst, table)
        finally:
            assert src.read_bytes() == source, "Original input was changed."
        # Header line endings may be LF or CRLF; sequence bytes must be exact.
        actual = dst.read_bytes().splitlines(keepends=True)
        expected = b"".join(expected_lines).splitlines(keepends=True)
        assert len(actual) == len(expected), "Line count/wrapping changed."
        for got, want in zip(actual, expected):
            if want.startswith(b">"):
                assert got.rstrip(b"\r\n") == want.rstrip(b"\r\n"), "Incorrect new ID/order."
            else:
                assert got == want, "A sequence line changed (case, whitespace, or newline)."
        with table.open(newline="", encoding="utf-8-sig") as handle:
            reader = csv.DictReader(handle)
            assert reader.fieldnames == ["new_id", "original_header"], "CSV column names/order differ."
            rows = list(reader)
        wanted = [{"new_id": f"seq{i:03d}", "original_header": h}
                  for i, h in enumerate(expected_headers, 1)]
        assert rows == wanted, "CSV rows do not preserve the original headers/order."
    return True

validation = {"status": "NOT READY", "checks": []}
try:
    verify_case(clean_fasta, EXAMPLE, ["sample A | gene=psbA", "sample A | gene=psbD"])
    # Regenerate submission files only after the isolated known-result test succeeds.
    before = INPUT.read_bytes()
    try:
        clean_fasta(INPUT, CLEANED, MAPPING)
    finally:
        assert INPUT.read_bytes() == before == EXAMPLE, "Course input changed."
    validation = {"status": "PASS", "checks": ["known result, sequence bytes, CSV mapping, unchanged input"]}
    print("PASS — known-result checks")
    print(CLEANED.read_text())
    print(MAPPING.read_text())
except Exception as error:
    validation = {"status": "NOT READY", "checks": [f"{type(error).__name__}: {error}"]}
    print("NOT READY:", validation["checks"][0])

### D1 result
Did the tests pass? If you revised code, what changed? **[Your answer]**

### D2. Choose one challenge
Ask one follow-up question: duplicate headers? lowercase sequences? empty input? accidental input overwrite? Choose **one** test below (or write your own). Extra cases are available for exploration; they are not extra graded tasks.

For empty input, this notebook uses the reasonable convention of an empty FASTA output and a CSV containing only the column header. Header lines may use LF or CRLF; sequence lines must retain their original bytes. These are exercise conventions, not a universal FASTA standard.

In [ ]:
CHALLENGE = "duplicates_and_lowercase"  # or "empty", "overwrite", "many_records"
try:
    if CHALLENGE == "duplicates_and_lowercase":
        source = (DATA / "challenge.fasta").read_bytes()
        verify_case(clean_fasta, source, ['same header, "quoted"'] * 2)
    elif CHALLENGE == "empty":
        verify_case(clean_fasta, b"", [])
    elif CHALLENGE == "many_records":
        source = b"".join(f">record {i}\nACGT\n".encode() for i in range(1001))
        verify_case(clean_fasta, source, [f"record {i}" for i in range(1001)])
    elif CHALLENGE == "overwrite":
        with tempfile.TemporaryDirectory() as folder:
            p = Path(folder) / "in.fasta"
            q = Path(folder) / "other.csv"
            for paths in [(p,p,q), (p,q,p), (p,q,q)]:
                p.write_bytes(EXAMPLE)
                q.write_bytes(b"sentinel")
                rejected = False
                try:
                    clean_fasta(*paths)
                except (ValueError, FileExistsError):
                    rejected = True
                assert p.read_bytes() == EXAMPLE and q.read_bytes() == b"sentinel", "Files changed before path validation."
                assert rejected, "Unsafe path combination should raise ValueError or FileExistsError."
    else:
        raise ValueError("Choose one of the listed challenge names.")
    print("PASS — selected challenge:", CHALLENGE)
except Exception as error:
    print("NOT READY — selected challenge:", type(error).__name__, str(error))

### D3. AI interaction log
- **AI assistant used:** [Your answer]
- **My original prompt:** [Paste your actual prompt]
- **My follow-up question:** [Paste your question]
- **What AI suggested (1–2 sentences):** [Your answer]
- **How I checked the result:** [Your answer; name the test and observed outcome]
- **One thing I learned:** [Your answer]

A good question and careful verification are sufficient; a complicated revision is unnecessary.

### Final reflection (included in D)
In 2–3 sentences, describe one advantage and one limitation or risk of using AI to write bioinformatics programs. **[Your answer]**

## Optional extension — not graded
Ask AI for a second function that saves sequence length and GC percentage to CSV. Test `data/gc_test.fasta`: `GCNN` must have length **4** and GC **50%**, using all four bases in the denominator. Why do N bases remain in the denominator under this definition? This is optional and does not affect your grade.

In [ ]:
# Optional workspace: paste and test your GC summary function here.
# The test input is DATA / "gc_test.fasta".
# Leave this cell blank if you skip the extension.

## E. Save and submit — 2 points (5 min)
The next cell validates the **current** code again before exporting, so old results cannot silently pass after a code change. It creates a ZIP only if the required known-result test passes. The ZIP contains the cleaned FASTA, header mapping CSV, original synthetic input, and a validation report. Your notebook is submitted separately.

Run export after your last code change. Then run the download cell (Colab only). Download your completed notebook using **File → Download → Download .ipynb** after saving all responses and outputs. Submit that `.ipynb` and `python_fasta_results.zip` through Canvas. Keep the displayed outputs. If you cannot resolve a failure, submit your notebook with the error and describe what you tried; reasonable attempts receive substantial credit.

In [ ]:
ZIP_PATH = ROOT / "python_fasta_results.zip"
# Remove an older archive so a failed rerun cannot leave a misleading export.
ZIP_PATH.unlink(missing_ok=True)
try:
    verify_case(clean_fasta, EXAMPLE, ["sample A | gene=psbA", "sample A | gene=psbD"])
    assert INPUT.read_bytes() == EXAMPLE, "Original course input changed; rerun setup."
    try:
        clean_fasta(INPUT, CLEANED, MAPPING)
    finally:
        assert INPUT.read_bytes() == EXAMPLE, "Input was changed."
    # Check the actual deliverables, independently of the isolated test.
    assert CLEANED.read_bytes().replace(b"\r\n", b"\n") == b">seq001\nATGCGT\nAATT\n>seq002\nGGCC\nATAT\n"
    with MAPPING.open(newline="", encoding="utf-8-sig") as handle:
        assert list(csv.DictReader(handle)) == [
            {"new_id": "seq001", "original_header": "sample A | gene=psbA"},
            {"new_id": "seq002", "original_header": "sample A | gene=psbD"}]
    report = {"status": "PASS", "scope": "Required two-record known-result checks only; not universal correctness",
              "input_sha256": hashlib.sha256(EXAMPLE).hexdigest()}
    report_path = RESULTS / "validation_report.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in [INPUT, CLEANED, MAPPING, report_path]:
            archive.write(path, path.relative_to(ROOT))
    print("Export ready:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        assert archive.testzip() is None
        print("ZIP contents:", archive.namelist())
except Exception as error:
    print("Export NOT READY:", type(error).__name__, str(error))

In [ ]:
if ZIP_PATH.exists():
    try:
        from google.colab import files
    except ImportError:
        print("Outside Colab: retrieve the ZIP from", ZIP_PATH)
    else:
        files.download(str(ZIP_PATH))
else:
    print("No ZIP to download yet. Resolve the required validation first.")

## Rubric and final checklist
| Component | Points |
|---|---:|
| A. Python experiments and explanations | 4 |
| B. Understanding the FASTA problem | 3 |
| C. AI-generated tool and explanation | 6 |
| D. Verification, follow-up question, AI log, and reflection | 5 |
| E. Complete, readable notebook and results ZIP | 2 |
| **Total** | **20** |

Grading emphasizes reasonable attempts, clear explanations, and evidence of checking, even if code is imperfect. Confirm that your notebook includes your actual prompts, responses, code, and executed outputs. AI helps write code; independent checks establish confidence in its scientific results.

Colab help: https://research.google.com/colaboratory/faq.html